### Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from spextractor import Spextractor
import json
import csv
import os

### Making .txt file from the .json file

In [16]:
json_file = '/home/rabbit/SpectralAnalysis_SNe/examples/ASASSN-14hr.json'
output_dir = '/home/rabbit/SpectralAnalysis_SNe/examples/processed/'
os.makedirs(output_dir, exist_ok=True)

with open(json_file, 'r') as f:
    data_structure = json.load(f)
for sn_name, sn_data in data_structure.items():
    spectra_list = sn_data['spectra']
    
    redshift = sn_data.get('redshift', [{}])[0]

    times = np.array([float(spec['time']) for spec in spectra_list])
    idx = np.argmin(times)
    closest_spec = spectra_list[idx]
    selected_mjd = float(closest_spec['time'])
    print(f"{sn_name}: selected spectrum at MJD {selected_mjd}, z={redshift}")

    raw_data = closest_spec['data']
    data_array = np.array(raw_data, dtype=float)
    wavelength = data_array[:, 0]
    flux = data_array[:, 1]
    error = data_array[:, 2]

    mask = np.isclose(flux, error)
    if np.any(mask):
        error[mask] = 1e-20 

    dat_file = os.path.join(output_dir, f"{sn_name}.dat")
    clean_data = np.column_stack((wavelength, flux, error))
    np.savetxt(dat_file, clean_data, fmt='%.6e', header='wavelength flux error')

    meta_file = os.path.join(output_dir, f"{sn_name}_meta.json")
    meta_info = {'MJD': selected_mjd, 'redshift': redshift}
    with open(meta_file, 'w') as mf:
        json.dump(meta_info, mf, indent=2)

    print(f"{sn_name}: saved {dat_file} and metadata {meta_file}")

ASASSN-14hr: selected spectrum at MJD 56933.0, z={'value': '0.03362', 'source': '3,6,7,9'}
ASASSN-14hr: saved /home/rabbit/SpectralAnalysis_SNe/examples/processed/ASASSN-14hr.dat and metadata /home/rabbit/SpectralAnalysis_SNe/examples/processed/ASASSN-14hr_meta.json


In [17]:
fn = '/home/rabbit/SpectralAnalysis_SNe/examples/processed/ASASSN-14hr.dat'
z = 0.03362
spex = Spextractor(fn, z=z, plot=True, log=True)

spex.create_model(model_type='gpr', downsampling=3.0)

features = ('Si II 6150A', 'Si II 5800A')
spex.process(features)

vsi = spex.vel[features[0]]
vsi_err = spex.vel_err[features[0]]
print(f'vsi = {vsi:.3f} +- {vsi_err:.3f}')

fig, ax = spex.fig_ax

plt.tight_layout()
fig.savefig('Ia_example.png', dpi=300)
plt.close('all')

vsi = 11.818 +- 0.619


/tmp/ipykernel_9585/1113871844.py:16: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
